# Final Evaluation
This Notebook contains the code for the final test evaluation and error analysis. Test metrics, confusion matrices, inspect missed failures and false alarms

### Imports

In [ ]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from src.data import CLASSES, MODES, PRIORITY, RAW_FEATURES, load_raw
from src.splits import load_split
from src.models_mlp import make_mlp
from src.models_rf import make_rf, make_class_weights, resolve_params
from src.evaluate import evaluate, evaluate_multi, metrics_at_threshold, metrics_at_cut_multi, summarize, summarize_multi, _fold_metrics, _fold_metrics_multi

import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

ROOT = pathlib.Path.cwd().parent
RESULTS, FIG = ROOT / "results", ROOT / "results" / "figures"

## Preprocessing Patch

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from src.data import NUMERIC, TYPES
import src.models_mlp as models_mlp

def make_preprocessor_dummies(scale=True):
    num = StandardScaler() if scale else "passthrough"
    return ColumnTransformer([
        ("num", num, NUMERIC),
        ("type", "passthrough", [f"Type_{t}" for t in TYPES]),
    ], verbose_feature_names_out=False)

models_mlp.make_preprocessor = make_preprocessor_dummies

## Fetching Final Hyperparameters

In [ ]:
best = json.load(open(RESULTS / "best_params.json"))[0]
rf_bin = json.load(open(RESULTS / "rf_binary_candidate.json"))
rf_multi = json.load(open(RESULTS / "rf_multi_candidate.json"))

FINAL = {  # decision = threshold (binary) or failure cut-off (multi), chosen on validation
    ("MLP", "binary"): {"params": best["mlp_bin"],   "decision": 0.25,                "features": None},
    ("RF",  "binary"): {"params": best["rf_bin"],    "decision": rf_bin["threshold"], "features": rf_bin["features"]},
    ("MLP", "multi"):  {"params": best["mlp_multi"], "decision": 0.40,                "features": None},
    ("RF",  "multi"):  {"params": best["rf_multi"],  "decision": rf_multi["cut"],     "features": rf_multi["features"]},
}

> These are the four configurations chosen at validation and are loaded from best_params.json and the RF candidate files. \
> Nothing is chosen here, the decision rule of each model was fixed on validation: a probability threshold for binary, and a failure cut-off for the two-stage multiclass rule. \
> The RF uses the 9 selected features and the MLP uses all 11, as in tuning.

## Helper Functions

In [ ]:
def build(model, task, y_train):
    p = FINAL[(model, task)]["params"]
    return make_mlp(**p) if model == "MLP" else make_rf(**resolve_params(p, make_class_weights(y_train)))

def predict(fitted, X, task, decision):
    """Same decision rules as in validation: threshold on P(failure), or two-stage cut-off for multi."""
    proba = fitted.predict_proba(X)
    if task == "binary":
        return (proba[:, 1] >= decision).astype(int), proba[:, 1]
    p_fail = 1 - proba[:, 0]
    return np.where(p_fail >= decision, 1 + proba[:, 1:].argmax(axis=1), 0), p_fail

## Scoring on Test Set

In [ ]:
splits = {t: load_split(t) for t in ["binary", "multi"]}
rows, preds, fitted = [], {}, {}

for (model, task), cfg in FINAL.items():
    X_tr, X_te, y_tr, y_te = splits[task]
    cols = cfg["features"] or list(X_tr.columns)
    X_tr, X_te, d = X_tr[cols], X_te[cols], cfg["decision"]

    if task == "binary":
        _, probas = evaluate(build(model, task, y_tr), X_tr, y_tr)
        val = summarize(metrics_at_threshold(probas, d))
    else:
        _, probas = evaluate_multi(build(model, task, y_tr), X_tr, y_tr)
        val = summarize_multi(metrics_at_cut_multi(probas, d))

    m = build(model, task, y_tr).fit(X_tr, y_tr)
    pred, score = predict(m, X_te, task, d)
    fitted[(model, task)], preds[(model, task)] = m, (pred, score)

    if task == "binary":
        te = _fold_metrics(y_te, pred)
        keys = {"accuracy": "accuracy", "precision": "precision", "recall": "recall", "f1": "f1"}
    else:
        te = _fold_metrics_multi(y_te, pred)
        keys = {"accuracy": "accuracy", "precision": "detection_precision", "recall": "detection_recall",
                "f1": "detection_f1", "f1_macro": "f1_macro"}
    for metric, k in keys.items():
        rows.append({"model": model, "task": task, "decision": d, "metric": metric,
                     "val_mean": val[f"{k}_mean"], "val_std": val[f"{k}_std"], "test": te[k]})

final = pd.DataFrame(rows)
final.to_csv(RESULTS / "final_metrics.csv", index=False)
final.round(3)

For each model, the validation metrics come from the same 5×3 repeated stratified CV on the training set used in tuning, reported as mean +/- std. The model is then trained once on the full training set and evaluated once on the hold-out test set.

The RF is the better model in both tasks (test F1): 
- **Random Forest**: 0.881 +/- 0.017 in binary and 0.871 +/- 0.020 in multi-class
- **MLP**: 0.671 +/- 0.077 in binary and 0.767 +/- 0.030 in multi-class

Three of the four models land within about one std of their validation score.\
**The exception:** MLP in binary, where recall drops from 0.749 to 0.612, about 1.8 std below. Its precision holds: 0.741 vs 0.743.\
A likely reason is that the MLP's recall varies a lot between folds, and the test set has only 85 failures, so each missed failure costs 1.2 points of recall.

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, ((model, task), (pred, _)) in zip(axes.ravel(), preds.items()):
    labels = ["No failure", "Failure"] if task == "binary" else CLASSES
    ConfusionMatrixDisplay.from_predictions(splits[task][3], pred, labels=range(len(labels)),
                                            display_labels=labels, ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(f"{model} – {task} (test)"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(FIG / "test_confusion_matrices.png", dpi=150); plt.show()

**Binary:** 
- The RF misses 15 of 85 failures with only 4 false positives. 
- The MLP misses 33 and raises 18 false alarms. 

**Multi-class:** 
- Both models identify HDF, PWF and OSF well (the RF gets almost all of them)
- TWF is almost never detected: 
  - 10 of 11 missed by the MLP
  - 9 missed plus 1 confused with OSF by the RF. 

Confusions between failure types are rare. Almost all errors are between "No failure" and some failure.

## Missed Failures Vs False Alarms

In [ ]:
change = []
for (model, task), cfg in FINAL.items():
    _, X_te, _, y_te = splits[task]
    X_te = X_te[cfg["features"] or list(X_te.columns)]
    for label, d in [("default", 0.5), ("chosen", cfg["decision"])]:
        pred, _ = predict(fitted[(model, task)], X_te, task, d)
        fail, p_fail = (y_te > 0).to_numpy(), pred > 0
        change.append({"model": model, "task": task, "decision": f"{label} ({d})",
                       "missed_failures": int((fail & ~p_fail).sum()), "false_alarms": int((~fail & p_fail).sum())})
pd.DataFrame(change)

Here the same trained models are scored at the default 0.5 and at the decision value chosen on validation. This shows what the threshold choice did on test\
Lowering the threshold trades false positives for fewer false negatives:
- **MLP binary:** from 44 to 33 FN 6 → 18 false alarms. 
- **RF multi:** 13 → 9 missed and 3 → 13 false alarms. 
- **MLP multi:** 24 → 22 missed and 8 → 15 false alarms. 

For RF binary, 0.4 and 0.5 give the same counts, because its probabilities are well separated around those values.
> Since we assumed a missed failure costs more than an inspection, these trades are acceptable.

## Error Analysis

### Binary Classification

In [ ]:
raw = load_raw()
_, X_te, _, y_te = splits["binary"]
(pred_mlp, p_mlp), (pred_rf, p_rf) = preds[("MLP", "binary")], preds[("RF", "binary")]
err = raw.loc[X_te.index].assign(y=y_te, p_mlp=p_mlp.round(3), p_rf=p_rf.round(3), mlp=pred_mlp, rf=pred_rf)

display(pd.DataFrame({"MLP missed": err[(err.y == 1) & (err.mlp == 0)][MODES].sum(),
                      "RF missed":  err[(err.y == 1) & (err.rf == 0)][MODES].sum(),
                      "all test failures": err[err.y == 1][MODES].sum()}))

show = ["UDI", "Type", *RAW_FEATURES, *MODES, "p_mlp", "p_rf", "mlp", "rf"]
display(err[(err.y == 1) & ((err.mlp == 0) | (err.rf == 0))][show])   # missed failures
display(err[(err.y == 0) & ((err.mlp == 1) | (err.rf == 1))][show])   # false alarms

> **Note:** The table counts modes, so a row with two modes is counted twice.

**Most of the missed failures are TWF:** 12 of 13 TWF failures are missed by both models, because tool-wear failures happen at a random point in a wear range that many healthy machines also reach. 

The 2 failures with no recorded mode are also missed by both, as expected, since nothing in the features explains them.\
Beyond that: 
- The RF misses only 1 HDF 
- The MLP misses 8 HDF, 3 PWF and 9 OSF, which are failures defined by sharp rules on temperature difference, power and strain. 

Trees capture threshold rules like these directly, while the MLP has to approximate them with smooth functions. The RF's 4 false positives all fall in conditions similar to real HDF failures (high air temperature, low speed).

### Multi-class Classification

In [ ]:
_, X_te, _, y_te = splits["multi"]
raw_te = raw.loc[X_te.index]
print("multi-mode rows in test:", int((raw_te[PRIORITY].sum(axis=1) > 1).sum()))
for model in ["MLP", "RF"]:
    pred, _ = preds[(model, "multi")]
    wrong = pred != y_te.to_numpy()
    other = np.array([p > 0 and raw_te.iloc[i][CLASSES[p]] == 1 for i, p in enumerate(pred)]) & wrong
    print(f"{model}: {other.sum()} of {wrong.sum()} test errors predicted another mode that is also present")

The test set has 5 rows with more than one active mode, which were labelled with their rarest mode. Only 1 of the MLP's 38 test errors, and none of the RF's 24, predict another mode that is also active on that row. So the labelling rule for multi-mode rows has almost no effect on the results, and nearly all errors are real misses or false positives.